In [6]:
import os
import getpass
from sqlalchemy import URL, create_engine, text

db_password = os.getenv("DB_PASSWORD") or getpass.getpass("MySQL password: ")

server_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER", "root"),
    password=db_password,
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "3306")),
)

server_engine = create_engine(server_url, pool_pre_ping=True)

with server_engine.connect() as conn:
    databases = conn.execute(text("SHOW DATABASES")).scalars().all()

print("Available databases:", databases)

Available databases: ['aggregation_test', 'data_cleansing', 'data_engineering', 'information_schema', 'multitenant_db', 'mysql', 'performance_schema', 'sales_analytics', 'sys', 'test_case_10', 'test_case_11', 'test_case_12', 'test_case_13', 'test_case_14', 'test_case_15_production', 'test_case_15_staging', 'test_case_16', 'test_case_2', 'test_case_2_dw', 'test_case_5', 'test_case_6', 'test_case_7', 'test_case_8', 'test_case_9', 'test_cases']


In [7]:
from sqlalchemy import URL, create_engine, text
import pandas as pd

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER", "root"),
    password=db_password,
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "3306")),
    database="aggregation_test",
)

engine = create_engine(connection_url, pool_pre_ping=True)

with engine.connect() as conn:
    tables = conn.execute(text("SHOW TABLES")).all()

print("Tables:", tables)

Tables: [('sales',), ('sales_partitioned',), ('sales_shard_1',), ('sales_shard_2',), ('sales_shard_3',)]


In [8]:
with engine.connect() as conn:
    columns = pd.read_sql(
        text("""
            SELECT COLUMN_NAME, DATA_TYPE
            FROM INFORMATION_SCHEMA.COLUMNS
            WHERE TABLE_SCHEMA = DATABASE()
              AND TABLE_NAME = 'sales'
            ORDER BY ORDINAL_POSITION
        """),
        conn,
    )

    sample_rows = pd.read_sql(
        text("SELECT * FROM sales LIMIT 5"),
        conn,
    )

display(columns)
display(sample_rows)

,COLUMN_NAME,DATA_TYPE
0,sale_id,int
1,sale_date,date
2,region,varchar
3,product_id,int
4,customer_id,int
5,quantity,int
6,unit_price,decimal
7,total_amount,decimal


,sale_id,sale_date,region,product_id,customer_id,quantity,unit_price,total_amount
0,1,2023-01-01,North,1,1,1,100.0,100.0
1,2,2023-01-02,South,2,2,2,101.0,202.0
2,3,2023-01-03,East,3,3,3,102.0,306.0
3,4,2023-01-04,West,4,4,4,103.0,412.0
4,5,2023-01-05,Central,5,5,5,104.0,520.0


In [9]:
anomaly_query = text("""
    WITH daily_sales AS (
        SELECT
            sale_date,
            region,
            SUM(total_amount) AS daily_revenue
        FROM sales
        GROUP BY sale_date, region
    ),
    regional_stats AS (
        SELECT
            region,
            AVG(daily_revenue) AS avg_revenue,
            STDDEV_SAMP(daily_revenue) AS stddev_revenue
        FROM daily_sales
        GROUP BY region
    ),
    scored_sales AS (
        SELECT
            d.sale_date,
            d.region,
            d.daily_revenue,
            s.avg_revenue,
            s.stddev_revenue,
            (d.daily_revenue - s.avg_revenue)
                / NULLIF(s.stddev_revenue, 0) AS z_score
        FROM daily_sales AS d
        JOIN regional_stats AS s
          ON d.region = s.region
    )
    SELECT
        sale_date,
        region,
        daily_revenue,
        ROUND(avg_revenue, 2) AS avg_revenue,
        ROUND(z_score, 2) AS z_score
    FROM scored_sales
    WHERE ABS(z_score) >= 3
    ORDER BY ABS(z_score) DESC
""")

with engine.connect() as conn:
    anomalies = pd.read_sql(anomaly_query, conn)

print("Anomalous region-days found:", len(anomalies))
display(anomalies.head(20))

Anomalous region-days found: 0


,sale_date,region,daily_revenue,avg_revenue,z_score


In [10]:
rolling_query = text("""
    WITH daily_sales AS (
        SELECT
            sale_date,
            region,
            SUM(total_amount) AS daily_revenue
        FROM sales
        GROUP BY sale_date, region
    ),
    rolling_baseline AS (
        SELECT
            sale_date,
            region,
            daily_revenue,
            COUNT(*) OVER (
                PARTITION BY region
                ORDER BY sale_date
                ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
            ) AS history_days,
            AVG(daily_revenue) OVER (
                PARTITION BY region
                ORDER BY sale_date
                ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
            ) AS avg_previous_7_days,
            STDDEV_SAMP(daily_revenue) OVER (
                PARTITION BY region
                ORDER BY sale_date
                ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
            ) AS stddev_previous_7_days
        FROM daily_sales
    ),
    scored AS (
        SELECT
            *,
            (daily_revenue - avg_previous_7_days)
                / NULLIF(stddev_previous_7_days, 0) AS rolling_z_score
        FROM rolling_baseline
        WHERE history_days = 7
    )
    SELECT
        sale_date,
        region,
        daily_revenue,
        ROUND(avg_previous_7_days, 2) AS avg_previous_7_days,
        ROUND(rolling_z_score, 2) AS rolling_z_score,
        CASE
            WHEN ABS(rolling_z_score) >= 3 THEN 'ANOMALY'
            ELSE 'NORMAL'
        END AS anomaly_status
    FROM scored
    WHERE stddev_previous_7_days > 0
    ORDER BY ABS(rolling_z_score) DESC
    LIMIT 20
""")

with engine.connect() as conn:
    rolling_results = pd.read_sql(rolling_query, conn)

display(rolling_results)

,sale_date,region,daily_revenue,avg_previous_7_days,rolling_z_score,anomaly_status
0,2024-03-05,West,27146.0,32913.43,-7.80,ANOMALY
1,2024-03-06,Central,31880.0,38002.86,-7.61,ANOMALY
2,2024-03-04,East,22438.0,27851.71,-7.51,ANOMALY
3,2026-11-29,East,22438.0,27692.29,-6.87,ANOMALY
4,2025-07-17,East,22438.0,27692.29,-6.87,ANOMALY
5,2024-03-03,South,17756.0,22817.71,-6.73,ANOMALY
6,2025-07-16,South,17756.0,22711.71,-6.59,ANOMALY
7,2026-11-28,South,17756.0,22711.71,-6.59,ANOMALY
8,2026-11-30,West,27146.0,32700.29,-6.53,ANOMALY
9,2025-07-18,West,27146.0,32700.29,-6.53,ANOMALY


In [11]:
anomaly_summary_query = text("""
    WITH daily_sales AS (
        SELECT
            sale_date,
            region,
            SUM(total_amount) AS daily_revenue
        FROM sales
        GROUP BY sale_date, region
    ),
    rolling_baseline AS (
        SELECT
            sale_date,
            region,
            daily_revenue,
            COUNT(*) OVER (
                PARTITION BY region
                ORDER BY sale_date
                ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
            ) AS history_days,
            AVG(daily_revenue) OVER (
                PARTITION BY region
                ORDER BY sale_date
                ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
            ) AS avg_previous_7_days,
            STDDEV_SAMP(daily_revenue) OVER (
                PARTITION BY region
                ORDER BY sale_date
                ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
            ) AS stddev_previous_7_days
        FROM daily_sales
    ),
    scored AS (
        SELECT
            region,
            daily_revenue,
            (daily_revenue - avg_previous_7_days)
                / NULLIF(stddev_previous_7_days, 0) AS rolling_z_score
        FROM rolling_baseline
        WHERE history_days = 7
          AND stddev_previous_7_days > 0
    )
    SELECT
        region,
        COUNT(*) AS evaluated_days,
        SUM(CASE WHEN ABS(rolling_z_score) >= 3 THEN 1 ELSE 0 END)
            AS anomaly_days,
        ROUND(MAX(ABS(rolling_z_score)), 2) AS largest_abs_z_score
    FROM scored
    GROUP BY region
    ORDER BY anomaly_days DESC, region
""")

with engine.connect() as conn:
    anomaly_summary = pd.read_sql(anomaly_summary_query, conn)

display(anomaly_summary)

,region,evaluated_days,anomaly_days,largest_abs_z_score
0,East,1454,57.0,7.51
1,South,1454,53.0,6.73
2,West,1454,53.0,7.80
3,Central,1454,45.0,7.61
4,North,1454,35.0,5.73


In [12]:
iqr_query = text("""
    WITH daily_sales AS (
        SELECT
            sale_date,
            region,
            SUM(total_amount) AS daily_revenue
        FROM sales
        GROUP BY sale_date, region
    ),
    ranked_sales AS (
        SELECT
            sale_date,
            region,
            daily_revenue,
            NTILE(4) OVER (
                PARTITION BY region
                ORDER BY daily_revenue
            ) AS quartile_group
        FROM daily_sales
    ),
    quartile_bounds AS (
        SELECT
            region,
            MAX(CASE WHEN quartile_group = 1 THEN daily_revenue END) AS q1,
            MIN(CASE WHEN quartile_group = 4 THEN daily_revenue END) AS q3
        FROM ranked_sales
        GROUP BY region
    ),
    fences AS (
        SELECT
            region,
            q1,
            q3,
            q3 - q1 AS iqr,
            q1 - 1.5 * (q3 - q1) AS lower_fence,
            q3 + 1.5 * (q3 - q1) AS upper_fence
        FROM quartile_bounds
    )
    SELECT
        d.sale_date,
        d.region,
        d.daily_revenue,
        ROUND(f.q1, 2) AS q1,
        ROUND(f.q3, 2) AS q3,
        ROUND(f.lower_fence, 2) AS lower_fence,
        ROUND(f.upper_fence, 2) AS upper_fence,
        CASE
            WHEN d.daily_revenue < f.lower_fence THEN 'LOW OUTLIER'
            WHEN d.daily_revenue > f.upper_fence THEN 'HIGH OUTLIER'
        END AS anomaly_type
    FROM daily_sales AS d
    JOIN fences AS f
      ON d.region = f.region
    WHERE d.daily_revenue < f.lower_fence
       OR d.daily_revenue > f.upper_fence
    ORDER BY d.region, d.sale_date
""")

with engine.connect() as conn:
    iqr_anomalies = pd.read_sql(iqr_query, conn)

print("IQR anomalies found:", len(iqr_anomalies))
display(iqr_anomalies.head(20))

IQR anomalies found: 5


,sale_date,region,daily_revenue,q1,q3,lower_fence,upper_fence,anomaly_type
0,2024-02-21,North,12670.0,15780.0,17780.0,12780.0,20780.0,LOW OUTLIER
1,2025-05-27,North,12670.0,15780.0,17780.0,12780.0,20780.0,LOW OUTLIER
2,2025-07-05,North,12670.0,15780.0,17780.0,12780.0,20780.0,LOW OUTLIER
3,2026-10-09,North,12670.0,15780.0,17780.0,12780.0,20780.0,LOW OUTLIER
4,2026-11-17,North,12670.0,15780.0,17780.0,12780.0,20780.0,LOW OUTLIER


In [13]:
detection_summary = pd.DataFrame(
    [
        {
            "method": "Regional Z-score",
            "anomaly_count": len(anomalies),
        },
        {
            "method": "Rolling 7-observation Z-score",
            "anomaly_count": int(anomaly_summary["anomaly_days"].sum()),
        },
        {
            "method": "IQR",
            "anomaly_count": len(iqr_anomalies),
        },
    ]
)

display(detection_summary)

,method,anomaly_count
0,Regional Z-score,0
1,Rolling 7-observation Z-score,243
2,IQR,5


In [14]:
# Re-run the rolling query without its LIMIT so the log gets every anomaly.
full_rolling_query = text(rolling_query.text.replace("LIMIT 20", ""))

with engine.connect() as conn:
    all_rolling_results = pd.read_sql(full_rolling_query, conn)

rolling_anomaly_log = all_rolling_results[
    all_rolling_results["anomaly_status"] == "ANOMALY"
].copy()

rolling_anomaly_log["detection_method"] = "Rolling 7-observation Z-score"
rolling_anomaly_log["severity"] = rolling_anomaly_log["rolling_z_score"].abs().apply(
    lambda score: "HIGH" if score >= 5 else "MEDIUM"
)

create_log_table = text("""
    CREATE TABLE IF NOT EXISTS anomaly_log (
        anomaly_id BIGINT AUTO_INCREMENT PRIMARY KEY,
        sale_date DATE NOT NULL,
        region VARCHAR(50) NOT NULL,
        detection_method VARCHAR(100) NOT NULL,
        daily_revenue DECIMAL(14, 2) NOT NULL,
        baseline_revenue DECIMAL(14, 2) NOT NULL,
        z_score DECIMAL(10, 2) NOT NULL,
        severity VARCHAR(10) NOT NULL,
        logged_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        UNIQUE KEY uq_anomaly_method_date_region
            (sale_date, region, detection_method)
    )
""")

insert_log_row = text("""
    INSERT INTO anomaly_log (
        sale_date,
        region,
        detection_method,
        daily_revenue,
        baseline_revenue,
        z_score,
        severity
    )
    VALUES (
        :sale_date,
        :region,
        :detection_method,
        :daily_revenue,
        :baseline_revenue,
        :z_score,
        :severity
    )
""")

with engine.begin() as conn:
    conn.execute(create_log_table)
    conn.execute(
        text("""
            DELETE FROM anomaly_log
            WHERE detection_method = 'Rolling 7-observation Z-score'
        """)
    )

    if not rolling_anomaly_log.empty:
        log_rows = [
            {
                "sale_date": row["sale_date"],
                "region": row["region"],
                "detection_method": row["detection_method"],
                "daily_revenue": float(row["daily_revenue"]),
                "baseline_revenue": float(row["avg_previous_7_days"]),
                "z_score": float(row["rolling_z_score"]),
                "severity": row["severity"],
            }
            for _, row in rolling_anomaly_log.iterrows()
        ]
        conn.execute(insert_log_row, log_rows)

print("Anomaly records logged:", len(rolling_anomaly_log))

Anomaly records logged: 243


In [15]:
with engine.connect() as conn:
    logged_summary = pd.read_sql(
        text("""
            SELECT
                detection_method,
                severity,
                COUNT(*) AS anomaly_count,
                MIN(sale_date) AS first_anomaly_date,
                MAX(sale_date) AS last_anomaly_date
            FROM anomaly_log
            GROUP BY detection_method, severity
            ORDER BY detection_method, severity
        """),
        conn,
    )

    logged_preview = pd.read_sql(
        text("""
            SELECT
                sale_date,
                region,
                daily_revenue,
                baseline_revenue,
                z_score,
                severity
            FROM anomaly_log
            ORDER BY ABS(z_score) DESC
            LIMIT 10
        """),
        conn,
    )

display(logged_summary)
display(logged_preview)

,detection_method,severity,anomaly_count,first_anomaly_date,last_anomaly_date
0,Rolling 7-observation Z-score,HIGH,16,2024-03-02,2026-12-01
1,Rolling 7-observation Z-score,MEDIUM,227,2023-01-27,2026-12-31


,sale_date,region,daily_revenue,baseline_revenue,z_score,severity
0,2024-03-05,West,27146.0,32913.43,-7.80,HIGH
1,2024-03-06,Central,31880.0,38002.86,-7.61,HIGH
2,2024-03-04,East,22438.0,27851.71,-7.51,HIGH
3,2025-07-17,East,22438.0,27692.29,-6.87,HIGH
4,2026-11-29,East,22438.0,27692.29,-6.87,HIGH
5,2024-03-03,South,17756.0,22817.71,-6.73,HIGH
6,2025-07-16,South,17756.0,22711.71,-6.59,HIGH
7,2026-11-28,South,17756.0,22711.71,-6.59,HIGH
8,2025-07-18,West,27146.0,32700.29,-6.53,HIGH
9,2026-11-30,West,27146.0,32700.29,-6.53,HIGH


In [16]:
with engine.connect() as conn:
    logged_count = conn.execute(
        text("""
            SELECT COUNT(*)
            FROM anomaly_log
            WHERE detection_method = 'Rolling 7-observation Z-score'
        """)
    ).scalar_one()

expected_count = len(rolling_anomaly_log)

assert logged_count == expected_count, (
    f"Log count mismatch: expected {expected_count}, found {logged_count}"
)

assert set(rolling_anomaly_log["severity"].dropna()).issubset(
    {"HIGH", "MEDIUM"}
)

print("Rolling anomalies detected:", expected_count)
print("Anomaly records verified:", logged_count)
print("TEST CASE 19 ANOMALY DETECTION VALIDATION PASSED")

Rolling anomalies detected: 243
Anomaly records verified: 243
TEST CASE 19 ANOMALY DETECTION VALIDATION PASSED
